# Diviser une Series par 60 : CPU, threads et GPU

**Question :** si chaque valeur peut être traitée indépendamment, comment
répartir le travail ? Le même calcul convertit des minutes en heures :
`hours = minutes / 60`.

Reprise du cours du 22 septembre, pour la partie CPU, threads et GPU
du cours du 24 septembre. Voir aussi le [README de ce dossier](README.md).
Ce notebook contient ses données et tout son code. Il compare **NumPy sur
CPU**, **plusieurs threads CPU** et, si disponible, **un GPU CUDA via
PyTorch**. Aucun module du cours ni téléchargement de données n'est requis.

**Dans Colab :** ouvrir [Colab](https://colab.research.google.com/), importer
ce seul fichier `.ipynb`, puis sélectionner un accélérateur **GPU** dans
les paramètres du type d'exécution (*Runtime / Change runtime type*).
Exécuter les cellules dans l'ordre. Le matériel attribué peut varier ;
vérifier le diagnostic ci-dessous. Sans GPU, la partie CPU reste exécutable.

**En local :** ouvrir avec Jupyter ou VS Code et un noyau disposant de
NumPy, pandas et Matplotlib. PyTorch et un GPU compatible CUDA sont
facultatifs. Ce notebook n'installe aucun paquet et ne change pas
l'environnement. Si PyTorch est absent, le signaler et poursuivre sur CPU.

## 1. Trois façons d'organiser le calcul

| Chemin | Organisation | Où se trouve le résultat ? |
|---|---|---|
| NumPy, un appel CPU | une routine compilée traite le tableau ; elle peut utiliser une instruction pour traiter plusieurs valeurs à la fois (SIMD) | mémoire du CPU |
| NumPy, plusieurs threads CPU | quelques blocs indépendants, une tâche par bloc | mémoire du CPU, partagée entre threads |
| PyTorch, GPU | un noyau de calcul répartit les valeurs entre de nombreux threads GPU | mémoire du GPU ; retour explicite vers le CPU |

Un **thread** est un fil d'exécution dans un processus. Les threads CPU
partagent sa mémoire ; le système peut les exécuter sur plusieurs cœurs.
**Créer quatre threads ne garantit pas quatre cœurs occupés ni un gain ×4.**

Le **GIL** est un verrou utilisé par certaines versions de Python.
Avec le **GIL activé**, CPython n'exécute qu'un thread de bytecode Python
à la fois. NumPy peut libérer ce verrou pendant le calcul natif sur des
tableaux numériques `float32` : nos blocs peuvent alors être calculés
en parallèle.
Cela ne s'applique pas automatiquement à une boucle Python ou à des objets
quelconques. Les versions de CPython sans GIL constituent un autre cas.
[Références Python](https://docs.python.org/3/library/threading.html#gil-and-performance-considerations)
et [NumPy](https://numpy.org/doc/stable/reference/thread_safety.html).

Les **threads GPU ne sont pas les threads de `ThreadPoolExecutor`**.
PyTorch lance un noyau compilé sur le GPU. Nous utilisons un seul appel
Python pour demander des milliers ou millions de divisions indépendantes.

In [ ]:
import os
import platform
import sys
from concurrent.futures import ThreadPoolExecutor
from threading import current_thread
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

print("Python:", platform.python_version(), platform.python_implementation())
print("Executable:", sys.executable)
print("System:", platform.platform())
print("CPU:", platform.processor() or platform.machine())
print("NumPy:", np.__version__, "pandas:", pd.__version__)
if hasattr(sys, "_is_gil_enabled"):
    print("GIL enabled:", sys._is_gil_enabled())

## 2. Paramètres et données communes

Commencer par **2 millions de valeurs**, soit 8 Mo par tableau `float32`.
Les CPU logiques disponibles ne sont pas forcément des cœurs physiques.
Nous plafonnons volontairement à quatre threads et au nombre de CPU
accessibles au processus. Si ce nombre vaut 1, nous ne pouvons pas
démontrer plusieurs cœurs avec ce notebook sur cette machine.

La préparation de la Series, sa conversion en tableau et le calcul de la
référence ont lieu **avant** le chronométrage. Aucune valeur manquante dans cet exemple.
Les comparaisons utilisent toutes les mêmes valeurs en `float32`.

In [ ]:
n_values = 2_000_000
requested_workers = 4
warmups = 3
repeats = 7

assert isinstance(n_values, int) and n_values > 0
assert isinstance(requested_workers, int) and requested_workers > 0
assert warmups >= 1 and repeats >= 1

if hasattr(os, "sched_getaffinity"):
    available_cpus = len(os.sched_getaffinity(0))
else:
    available_cpus = os.cpu_count() or 1
workers = min(requested_workers, available_cpus, 4, n_values)
print("Available logical CPUs:", available_cpus, "| CPU workers:", workers)

minutes = pd.Series(
    (np.arange(n_values) % 3600).astype(np.float32),
    name="resolution_minutes",
)
values = minutes.to_numpy(dtype=np.float32, copy=True)
values.setflags(write=False)
divisor = np.float32(60.0)
expected = (minutes / divisor).to_numpy(dtype=np.float32, copy=True)

display(pd.DataFrame({"minutes": minutes.head(), "hours": expected[:5]}))
print("Input:", values.shape, values.dtype, f"{values.nbytes / 1e6:.1f} MB")

## 3. CPU : un appel NumPy, puis plusieurs blocs

`out=output` indique où NumPy doit écrire son résultat. Chaque appel crée
un tableau de sortie neuf. Le tableau d'entrée reste en lecture seule.
Nous n'effectuons aucune modification concurrente de la Series pandas.

In [ ]:
def divide_numpy():
    output = np.empty_like(values)
    np.divide(values, divisor, out=output)
    return output

np.testing.assert_allclose(divide_numpy(), expected, rtol=1e-6, atol=1e-6)

Découper **les indices**, puis affecter chaque intervalle `[start:stop]`
à une tâche. Les intervalles se suivent, sans trou ni chevauchement ;
ils couvrent aussi correctement une longueur non divisible par le nombre
de threads. Chaque thread écrit uniquement dans sa propre tranche.

In [ ]:
edges = [index * n_values // workers for index in range(workers + 1)]
bounds = list(zip(edges[:-1], edges[1:]))
display(pd.DataFrame(bounds, columns=["start", "stop"]).assign(
    size=[stop - start for start, stop in bounds]
))

def divide_chunk(source, output, start, stop):
    np.divide(source[start:stop], divisor, out=output[start:stop])
    return current_thread().name

def divide_threaded(pool):
    output = np.empty_like(values)
    futures = [
        pool.submit(divide_chunk, values, output, start, stop)
        for start, stop in bounds
    ]
    for future in futures:
        future.result()
    return output

`submit()` confie un appel à l'exécuteur. `future.result()` attend sa fin
et transmet une éventuelle erreur. Le `with` ferme les threads même si
le calcul échoue. Ici, le chronomètre inclura la soumission des tâches et
leur attente, mais pas la création/fermeture du groupe de threads.
[Documentation `ThreadPoolExecutor`](https://docs.python.org/3/library/concurrent.futures.html#concurrent.futures.ThreadPoolExecutor).

## 4. Mesurer et vérifier

Trois appels de chauffe, puis sept appels mesurés. **Chaque résultat est
vérifié**, y compris pendant la chauffe. Les assertions, conversions de
contrôle et affichages restent hors du chronomètre. Une petite tolérance
accepte les différences d'arrondi entre calculs `float32`.

Les deux chemins CPU incluent l'allocation du résultat. La création du
groupe de threads et ses premiers appels sont hors des mesures retenues.
L'ordre des méthodes peut influencer le cache et la charge de la machine :
refaire l'expérience avant de conclure à une différence minime.

**Comparaison avec le kit du 24 septembre :** dans
`../performance/compare_execution.py`, la création et la fermeture du
groupe de threads sont comprises dans le temps mesuré. Ici, elles en
sont exclues. Les deux durées ne mesurent donc pas exactement le même travail.

In [ ]:
def check_cpu(result):
    assert result.shape == expected.shape
    assert result.dtype == np.float32
    np.testing.assert_allclose(result, expected, rtol=1e-6, atol=1e-6)

def measure(operation, check, synchronize=None):
    durations = []
    for run in range(warmups + repeats):
        if synchronize is not None:
            synchronize()
        started = perf_counter()
        result = operation()
        if synchronize is not None:
            synchronize()
        elapsed_ms = (perf_counter() - started) * 1000
        check(result)
        if run >= warmups:
            durations.append(elapsed_ms)
        del result
    return np.asarray(durations)

timings = {}
timings["CPU · NumPy"] = measure(divide_numpy, check_cpu)

with ThreadPoolExecutor(max_workers=workers, thread_name_prefix="division") as pool:
    # A diagnostic run, outside the benchmark.
    diagnostic_output = np.empty_like(values)
    tasks = [
        pool.submit(divide_chunk, values, diagnostic_output, start, stop)
        for start, stop in bounds
    ]
    observed_threads = sorted({task.result() for task in tasks})
    check_cpu(diagnostic_output)
    print("Threads observed in the diagnostic run:", observed_threads)
    del diagnostic_output, tasks

    def run_threads():
        return divide_threaded(pool)

    timings[f"CPU · {workers} thread(s)"] = measure(run_threads, check_cpu)

print("CPU checks passed; the thread pool is closed.")

Le diagnostic affiche les threads qui ont pris des tâches sur **cet
appel**. Un thread peut traiter plusieurs blocs si ceux-ci sont courts ;
cette liste ne prouve ni l'occupation simultanée de plusieurs cœurs ni le
nombre exact de threads utilisés dans les répétitions suivantes.

## 5. GPU : localiser les données, puis lancer le calcul

Le GPU dispose de sa propre mémoire. Pour un tableau initialement sur le
CPU, le parcours est **CPU → copie vers GPU → division → copie vers CPU**.
Activer un GPU dans Colab ne déplace pas automatiquement pandas ou NumPy
sur ce GPU : il faut choisir une bibliothèque et y transférer les données.

Le code suivant vérifie réellement la présence de PyTorch et de CUDA.
**Aucune durée GPU ne sera produite si le GPU est absent.**

In [ ]:
try:
    import torch
except ImportError:
    torch = None

gpu_available = torch is not None and torch.cuda.is_available()
if torch is None:
    print("GPU skipped: PyTorch is not installed in this environment.")
else:
    print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda)
    print("CUDA available:", gpu_available)
    if gpu_available:
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("GPU skipped: no CUDA GPU is accessible to this kernel.")

Nous allons distinguer deux questions :

1. **Données déjà sur GPU :** combien coûte une division avec allocation
   du résultat, lancement et attente du noyau de calcul ?
2. **Données sur CPU, résultat attendu sur CPU :** combien coûte le trajet
   complet avec les deux transferts ?

Les appels CUDA sont souvent asynchrones : Python peut reprendre avant
la fin du calcul. `torch.cuda.synchronize()` **avant et après** la zone
mesurée empêche de ne chronométrer que la mise en file d'attente.
Nos mesures sont des durées écoulées vues du CPU, pas des temps GPU purs.
[PyTorch — exécution asynchrone](https://docs.pytorch.org/docs/stable/notes/cuda.html#asynchronous-execution)
et [synchronisation](https://docs.pytorch.org/docs/stable/generated/torch.cuda.synchronize.html).

In [ ]:
# Remove previous GPU measurements if this cell is rerun without a GPU.
for name in list(timings):
    if name.startswith("GPU"):
        del timings[name]

if gpu_available:
    # Preparation excluded from all timings: host tensor and resident GPU input.
    host_tensor = torch.from_numpy(values.copy())
    gpu_input = host_tensor.to("cuda")
    torch.cuda.synchronize()

    def divide_gpu_resident():
        output = torch.empty_like(gpu_input)
        torch.div(gpu_input, 60.0, out=output)
        return output

    def divide_gpu_roundtrip():
        device_input = host_tensor.to("cuda")
        device_output = torch.empty_like(device_input)
        torch.div(device_input, 60.0, out=device_output)
        return device_output.cpu()

    def check_tensor(result):
        assert tuple(result.shape) == expected.shape
        assert result.dtype == torch.float32
        # This validation transfer happens after the timer has stopped.
        check_cpu(result.detach().cpu().numpy())

    timings["GPU · déjà en mémoire GPU"] = measure(
        divide_gpu_resident, check_tensor, torch.cuda.synchronize
    )
    timings["GPU · aller + calcul + retour"] = measure(
        divide_gpu_roundtrip, check_tensor, torch.cuda.synchronize
    )
    print("GPU checks passed for resident data and CPU-to-CPU round trips.")

Les transferts peuvent coûter davantage que cette division très simple.
Les mesurer séparément aide à lire le résultat précédent. Nous mesurons
ici deux copies indépendantes : **ce ne sont pas des étapes mesurées à
l'intérieur du trajet complet**. Leur somme ne reproduit donc pas
exactement son temps (variabilité, allocations et synchronisations).

In [ ]:
transfer_timings = {}
if gpu_available:
    def copy_to_gpu():
        return host_tensor.to("cuda")

    def copy_to_cpu():
        return gpu_input.cpu()

    def check_transfer(result):
        np.testing.assert_array_equal(result.cpu().numpy(), values)

    transfer_timings["CPU → GPU (copie seule)"] = measure(
        copy_to_gpu, check_transfer, torch.cuda.synchronize
    )
    transfer_timings["GPU → CPU (copie seule)"] = measure(
        copy_to_cpu, check_transfer, torch.cuda.synchronize
    )

## 6. Lire les résultats selon le périmètre mesuré

| Ligne | Ce que le chronomètre inclut |
|---|---|
| CPU NumPy | allocation + division sur tout le tableau |
| CPU threads | allocation + soumission/attente des tâches + divisions par blocs |
| GPU déjà en mémoire GPU | allocation GPU + lancement + division + attente ; résultat sur GPU |
| GPU aller + calcul + retour | transfert initial + allocation/division GPU + transfert du résultat ; résultat sur CPU |

Le calcul GPU avec données déjà présentes sur le GPU répond à une autre question que le
trajet complet. **Pour remplacer une opération pandas/NumPy isolée, lire
surtout le trajet complet.** Garder plusieurs calculs et leurs données
sur le GPU peut rendre les transferts moins fréquents.

Les allocations ont lieu à chaque appel ; les allocateurs peuvent réutiliser
de la mémoire déjà réservée après la chauffe. Les temps dépendent de la
machine, du GPU, de la taille, des caches et de la charge du moment.

In [ ]:
def summarize(records):
    return pd.DataFrame([
        {
            "method": name,
            "min_ms": float(np.min(samples)),
            "median_ms": float(np.median(samples)),
            "max_ms": float(np.max(samples)),
            "repeats": len(samples),
        }
        for name, samples in records.items()
    ]).set_index("method")

summary = summarize(timings)
display(summary.round(4))
if transfer_timings:
    print("Independent transfer measurements (not added to the chart):")
    display(summarize(transfer_timings).round(4))

positions = np.arange(len(summary))
median = summary["median_ms"].to_numpy()
errors = np.vstack([
    median - summary["min_ms"].to_numpy(),
    summary["max_ms"].to_numpy() - median,
])
fig, ax = plt.subplots(figsize=(10, 2.3 + 0.5 * len(summary)))
ax.errorbar(median, positions, xerr=errors, fmt="o", capsize=5, color="#176087")
ax.set_yticks(positions, summary.index)
ax.invert_yaxis()
ax.set_xscale("log")
ax.set_xlabel("Durée (ms, échelle logarithmique) — point : médiane ; trait : min–max")
ax.set_title(f"Diviser {n_values:,} valeurs float32 par 60".replace(",", " "))
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
plt.show()
plt.close(fig)

## 7. Explorer sans changer la question

Avant chaque essai, prédire le résultat ; modifier les paramètres de la
section 2, puis **tout réexécuter** pour recréer données et fonctions.

- Comparer `n_values = 20_000`, `2_000_000`, puis `10_000_000` si la mémoire
  le permet. Vérifier si le classement change.
- Comparer `requested_workers = 1`, `2`, puis `4`. Noter le nombre de
  threads réellement retenu, plafonné au matériel accessible.
- Essayer `n_values = 10_003` et `requested_workers = 3` : toutes les
  valeurs sont-elles traitées malgré des blocs de tailles différentes ?
- Expliquer pourquoi un calcul GPU rapide peut coexister avec un trajet
  complet plus lent que NumPy. Où commencent et finissent les mesures ?

Une division lit une valeur et écrit un résultat pour très peu de calcul.
L'accès à la mémoire, le lancement des tâches et les transferts peuvent
dominer. **Un ralentissement est un résultat utile**, pas une raison de
supprimer la mesure. Le multithreading, la vectorisation et le calcul GPU
sont des mécanismes distincts ; aucun ne garantit un gain pour toute tâche.

## Sources techniques

- [Python — threads et GIL](https://docs.python.org/3/library/threading.html#gil-and-performance-considerations)
- [Python — ThreadPoolExecutor](https://docs.python.org/3/library/concurrent.futures.html#concurrent.futures.ThreadPoolExecutor)
- [NumPy — sécurité des threads et libération du GIL](https://numpy.org/doc/stable/reference/thread_safety.html)
- [PyTorch — CUDA, copies et asynchronisme](https://docs.pytorch.org/docs/stable/notes/cuda.html)
- [PyTorch — synchronize](https://docs.pytorch.org/docs/stable/generated/torch.cuda.synchronize.html)
- [PyTorch — partage mémoire avec NumPy](https://docs.pytorch.org/docs/stable/generated/torch.from_numpy.html)